In [21]:
from pyspark.sql import SparkSession
import os

# Force correct Python usage
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

spark = SparkSession.builder \
    .master("local[1]") \
    .appName("FintechRecommender") \
    .config("spark.sql.shuffle.partitions", "1") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print("✅ Spark ready")
print(f"Spark version: {spark.version}")

✅ Spark ready
Spark version: 4.1.1


In [22]:
import pandas as pd
import numpy as np
import mlflow
import warnings
warnings.filterwarnings('ignore')

from pyspark.ml.recommendation import ALS
from pyspark.sql.types import IntegerType, FloatType

print("✅ All imports successful!")

✅ All imports successful!


In [24]:
import sys
sys.path.append('../utils')
from snowflake_connector import read_table

# Load from Snowflake Gold layer
user_item_matrix = read_table('GOLD_USER_ITEM_MATRIX')
catalog_df = read_table('SILVER_PRODUCT_CATALOG', schema='GOLD_SILVER')

print("\n✅ Data loaded from Snowflake Gold!")
print(f"User-item matrix shape: {user_item_matrix.shape}")
print(f"Catalog shape: {catalog_df.shape}")
print("\nUser-item matrix sample:")
print(user_item_matrix.head(3))

✅ Loaded GOLD_USER_ITEM_MATRIX: 10,000 rows
✅ Loaded SILVER_PRODUCT_CATALOG: 8 rows

✅ Data loaded from Snowflake Gold!
User-item matrix shape: (10000, 9)
Catalog shape: (8, 7)

User-item matrix sample:
  CUSTOMER_ID  CHECKING_ACCOUNT  SAVINGS_ACCOUNT  TRAVEL_CREDIT_CARD  \
0     C_00001                 1                1                   0   
1     C_00002                 1                0                   1   
2     C_00003                 1                0                   0   

   CASHBACK_CREDIT_CARD  PERSONAL_LOAN  HOME_LOAN  INVESTMENT_FUND  \
0                     1              0          0                0   
1                     0              0          0                0   
2                     1              0          1                0   

   FIXED_DEPOSIT  
0              0  
1              0  
2              1  


In [25]:
# Product columns from Snowflake (uppercase)
product_columns = [
    'CHECKING_ACCOUNT', 'SAVINGS_ACCOUNT', 'TRAVEL_CREDIT_CARD',
    'CASHBACK_CREDIT_CARD', 'PERSONAL_LOAN', 'HOME_LOAN',
    'INVESTMENT_FUND', 'FIXED_DEPOSIT'
]

# Create integer mappings (ALS needs integers not strings)
customer_map = {cid: idx for idx, cid in enumerate(user_item_matrix['CUSTOMER_ID'].unique())}
product_map = {col: idx for idx, col in enumerate(product_columns)}

# Reverse maps for decoding later
reverse_customer_map = {v: k for k, v in customer_map.items()}
reverse_product_map = {v: k for k, v in product_map.items()}

# Melt matrix from wide to long format
user_item_long = user_item_matrix.melt(
    id_vars=['CUSTOMER_ID'],
    value_vars=product_columns,
    var_name='product_name',
    value_name='rating'
)

# Keep only owned products (rating=1)
user_item_long = user_item_long[user_item_long['rating'] == 1].copy()

# Map to integers
user_item_long['user_id'] = user_item_long['CUSTOMER_ID'].map(customer_map).astype(int)
user_item_long['item_id'] = user_item_long['product_name'].map(product_map).astype(int)
user_item_long['rating'] = 1.0

print("✅ Data prepared for ALS!")
print(f"Total interactions: {len(user_item_long):,}")
print(user_item_long[['user_id', 'item_id', 'rating']].head())

✅ Data prepared for ALS!
Total interactions: 27,589
   user_id  item_id  rating
0        0        0     1.0
1        1        0     1.0
2        2        0     1.0
3        3        0     1.0
4        4        0     1.0


In [26]:
# Convert to Spark DataFrame
spark_df = spark.createDataFrame(
    user_item_long[['user_id', 'item_id', 'rating']]
)

# Cast to correct types
spark_df = spark_df \
    .withColumn('user_id', spark_df['user_id'].cast(IntegerType())) \
    .withColumn('item_id', spark_df['item_id'].cast(IntegerType())) \
    .withColumn('rating', spark_df['rating'].cast(FloatType()))

# Split train/test
train_df, test_df = spark_df.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()

print("✅ Spark DataFrame ready!")
spark_df.show(5)

✅ Spark DataFrame ready!
+-------+-------+------+
|user_id|item_id|rating|
+-------+-------+------+
|      0|      0|   1.0|
|      1|      0|   1.0|
|      2|      0|   1.0|
|      3|      0|   1.0|
|      4|      0|   1.0|
+-------+-------+------+
only showing top 5 rows


In [29]:
import mlflow
mlflow.set_tracking_uri('file:///C:/Users/Foram/Projects/Fintech Recommender System/fintech-recommender/mlflow')
mlflow.set_experiment("fintech_recommender_als")

mlflow.set_experiment("fintech_recommender_als")

with mlflow.start_run(run_name="ALS_v2_snowflake"):

    als = ALS(
        maxIter=10,
        regParam=0.1,
        rank=10,
        userCol="user_id",
        itemCol="item_id",
        ratingCol="rating",
        implicitPrefs=True,
        coldStartStrategy="drop"
    )

    print("Training ALS model...")
    model = als.fit(train_df)

    # Log to MLflow
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("regParam", 0.1)
    mlflow.log_param("rank", 10)
    mlflow.log_param("data_source", "Snowflake Gold")
    mlflow.log_param("training_rows", train_df.count())

    print("✅ ALS model trained!")
    print(f"Run ID: {mlflow.active_run().info.run_id}")

Traceback (most recent call last):
  File "C:\Users\Foram\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "C:\Users\Foram\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "C:\Users\Foram\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "C:\Users\Foram\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\store\tracking\file_store.py", line 1663, in _read_helper
    result = read_yaml(root, file_name)
  File "C:\Users\Foram\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\utils\yaml_utils.py", line 104, in read_yaml
    r

Training ALS model...
✅ ALS model trained!
Run ID: c264cd540580405487c1131347f7cfea


In [30]:
# Generate top 3 recommendations per customer
recommendations = model.recommendForAllUsers(3)
recs_pandas = recommendations.toPandas()

# Decode back to original IDs
results = []
for _, row in recs_pandas.iterrows():
    customer_id = reverse_customer_map[row['user_id']]
    for rec in row['recommendations']:
        product_col = reverse_product_map[rec['item_id']]
        # Convert column name to product name
        product_name = product_col.lower()
        results.append({
            'customer_id': customer_id,
            'product_name': product_name,
            'als_score': round(rec['rating'], 4)
        })

als_results_df = pd.DataFrame(results)

print("✅ Recommendations generated!")
print(f"Total: {len(als_results_df):,}")
print("\nSample:")
print(als_results_df.head(12))

✅ Recommendations generated!
Total: 28,920

Sample:
   customer_id        product_name  als_score
0      C_00001     savings_account     0.9451
1      C_00001    checking_account     0.9353
2      C_00001       fixed_deposit     0.0391
3      C_00002  travel_credit_card     0.9440
4      C_00002    checking_account     0.9201
5      C_00002     savings_account     0.0567
6      C_00003           home_loan     0.9427
7      C_00003    checking_account     0.9167
8      C_00003     savings_account     0.0553
9      C_00004     investment_fund     0.9427
10     C_00004    checking_account     0.9195
11     C_00004     savings_account     0.0587


In [31]:
# Save recommendations
als_results_df.to_csv('../data/als_recommendations.csv', index=False)

# Log metrics to MLflow
mlflow.log_metric("total_recommendations", len(als_results_df))
mlflow.log_metric("unique_customers", als_results_df['customer_id'].nunique())
mlflow.end_run()

print("✅ ALS notebook complete!")
print(f"Saved: als_recommendations.csv")
print(f"Total recommendations: {len(als_results_df):,}")
print(f"Unique customers: {als_results_df['customer_id'].nunique():,}")

✅ ALS notebook complete!
Saved: als_recommendations.csv
Total recommendations: 28,920
Unique customers: 9,640
